# Imports/Settings

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
# 1. Стандартная библиотека
import os
import sys
import warnings
from pathlib import Path
import joblib

# --- ДИНАМИЧЕСКИЙ РАСЧЕТ АБСОЛЮТНЫХ ПУТЕЙ ---
notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "notebooks":
    PROJECT_ROOT = notebook_dir.parent
else:
    PROJECT_ROOT = notebook_dir

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Сторонние библиотеки
import pandas as pd
import gradio as gr
from hydra import initialize, compose

# 3. Локальные модули
from core.models import get_model
from core.utils import load_hydra_config

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
load_hydra_config.cache_clear()
cfg = load_hydra_config()

print(f"Проект: {cfg.project_name} | Режим: Error Analysis")

Проект: outsource_project_name | Режим: Error Analysis


# Artefacts Loading

In [ ]:
models_dir = Path(cfg.paths.root_dir) / cfg.paths.models_dir
version = cfg.model.version
model_name = cfg.model.name

print("Поднятие модели в память...")
prep = joblib.load(models_dir / f"preprocessing_v{version}.pkl")

model = get_model(cfg)
model.load(str(models_dir / f"{model_name}_v{version}.cbm"))

print("Модель готова к инференсу!")

# Prediction Function

In [ ]:
def predict_conversion(total_hits, visit_hour, is_weekend, device_category, browser):
    """
    Принимает данные из UI, возвращает вероятность целевого действия.
    """
    # 1. Собираем данные в формат Pandas (ровно так, как ждет пайплайн)
    input_data = pd.DataFrame([{
        'total_hits': total_hits,
        'visit_hour': visit_hour,
        'is_weekend': int(is_weekend),
        'device_category': device_category,
        'browser': browser
    }])

    # 2. Очистка и генерация фичей (наш умный Pipeline)
    clean_data = prep.transform(input_data)

    # 3. Предсказание
    if hasattr(model, 'predict_proba') and cfg.data.tabular.task_type == 'binary':
        prob = model.predict_proba(clean_data)[0][1]

        # Форматируем для красивого вывода
        result_text = f"🔥 Вероятность конверсии: {prob * 100:.1f}%\n"
        if prob > 0.5:
            result_text += "Вердикт: Высокий потенциал (Горячий лид)"
        else:
            result_text += "Вердикт: Низкий потенциал"

        return result_text
    else:
        # Для регрессии
        pred = model.predict(clean_data)[0]
        return f"📈 Предсказанное значение: {pred:.2f}"

# Gradio

In [ ]:
# Описываем входы (слайдеры, чекбоксы, дропдауны)
inputs = [
    gr.Slider(minimum=0, maximum=200, step=1, value=5, label="Количество действий (total_hits)"),
    gr.Slider(minimum=0, maximum=23, step=1, value=12, label="Час визита (visit_hour)"),
    gr.Checkbox(label="Выходной день? (is_weekend)"),
    gr.Dropdown(choices=["mobile", "desktop", "tablet"], value="mobile", label="Тип устройства"),
    gr.Dropdown(choices=["Chrome", "Safari", "Firefox", "Edge"], value="Chrome", label="Браузер")
]

# Описываем выход (Текстовое поле)
output = gr.Textbox(label="Результат модели ML")

# Собираем приложение
demo = gr.Interface(
    fn=predict_conversion,
    inputs=inputs,
    outputs=output,
    title=f"🔮 {cfg.project_name} - Демонстрация Модели",
    description="Изменяйте параметры сессии ниже, чтобы увидеть, как нейросеть оценивает вероятность конверсии в реальном времени.",
    theme="default"
)

# Запускаем! (Интерфейс появится прямо под этой ячейкой)
demo.launch(share=False)
# Если поставить share=True, Gradio даст публичную ссылку на 72 часа,
# которую можно отправить заказчику в Telegram!